In [ ]:
# silver.py
from datetime import datetime
import zipfile
from io import BytesIO
import pandas as pd
from minio import Minio
from delta import configure_spark_with_delta_pip
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, trim, when, lit, current_timestamp, rlike, col, lower
from pyspark.sql.types import DoubleType, IntegerType,StringType
from config import MinIOConfig
import logging
from typing import List, Dict, Optional


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)


class SilverProcessor:
    """Classe base para processar dados da camada Bronze → Silver"""
    
    def __init__(self, minio_config: MinIOConfig = None):
        # Isso é só guardar as informações
        self.minio_config = minio_config or MinIOConfig()
        # Isso é abrir a conexão de fato
        self.minio_client = Minio(
            self.minio_config.endpoint,
            access_key=self.minio_config.access_key,
            secret_key=self.minio_config.secret_key,
            secure=self.minio_config.secure
        )
        self.spark = self._create_spark_session()
        self._ensure_bucket_exists()
    
    def _create_spark_session(self) -> SparkSession:
        """Cria sessão Spark com Delta Lake e S3 configurados"""
        
        # Configurar Spark Builder
        builder = (
            # Inicia o construtor da sessão Spark
            SparkSession.builder

                # Nome da aplicação exibida na interface do Spark
                .appName("CVM_Silver_Processing")

                # ------------------------------------------------------------
                # Dependências externas que o Spark precisa baixar automaticamente
                # ------------------------------------------------------------
                .config(
                    "spark.jars.packages",
                    "io.delta:delta-core_2.12:2.4.0"  # Só Delta
                )
                .config(
                    "spark.jars",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar,/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.driver.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )
                .config(
                    "spark.executor.extraClassPath",
                    "/opt/spark-jars/hadoop-aws-3.3.4.jar:/opt/spark-jars/aws-java-sdk-bundle-1.12.262.jar"
                )

                # ------------------------------------------------------------
                # Ativação do Delta Lake no Spark
                # ------------------------------------------------------------

                # Liga extensões SQL específicas do Delta Lake
                .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")

                # Torna o catálogo Delta o padrão para criação/leitura de tabelas
                .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")

                # ------------------------------------------------------------
                # Configurações para acessar o MinIO usando S3A
                # ------------------------------------------------------------

                # Endereço HTTP do serviço MinIO
                .config("spark.hadoop.fs.s3a.endpoint", f"http://{self.minio_config.endpoint}")

                # Access key (usuário público) do MinIO
                .config("spark.hadoop.fs.s3a.access.key", self.minio_config.access_key)

                # Secret key (senha) do MinIO
                .config("spark.hadoop.fs.s3a.secret.key", self.minio_config.secret_key)

                # Obrigatório para MinIO: usa estilo path, não VHost
                .config("spark.hadoop.fs.s3a.path.style.access", "true")

                # Define explicitamente o filesystem S3A correto
                .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")

                # Desativa SSL porque MinIO normalmente roda em HTTP no ambiente local
                .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

                # Usa provedor simples de credenciais (ideal para MinIO)
                .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                        "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")

                # ------------------------------------------------------------
                # Configurações de performance do Spark
                # ------------------------------------------------------------

                # Ativa otimização adaptativa → Spark ajusta planos de execução dinamicamente
                .config("spark.sql.adaptive.enabled", "true")

                # Número de partições usadas em operações como join e groupBy
                .config("spark.sql.shuffle.partitions", "8")

                # Tamanho máximo de cada partição de arquivo (128 MB)
                # Evita gerar muitos arquivos pequenos
                .config("spark.sql.files.maxPartitionBytes", 134217728)
        )
        
        # Adicionar Delta Lake
        spark = configure_spark_with_delta_pip(builder).getOrCreate()
        spark.sparkContext.setLogLevel("WARN")
        
        logger.info("✅ Spark Session criada com Delta Lake")
        return spark
    
    def _ensure_bucket_exists(self):
        """Garante que o bucket silver existe"""
        if not self.minio_client.bucket_exists(self.minio_config.bucket_silver):
            self.minio_client.make_bucket(self.minio_config.bucket_silver)
            logger.info(f"Bucket {self.minio_config.bucket_silver} criado")


class CVMSilverProcessor(SilverProcessor):
    """Processa dados da CVM da camada Bronze para Silver usando Delta Lake"""
    
    # Definir quais CSVs queremos extrair do ZIP
    CSV_TARGETS = {
        "DRE_con": "dfp_cia_aberta_DRE_con",    # DRE Consolidado
        "DFC_DMPL_con": "dfp_cia_aberta_DMPL_con",  # Fluxo de Caixa Método Direto
        "BPA_con": "dfp_cia_aberta_BPA_con",     # Balanço Patrimonial Ativo Consolidado
        "BPP_con": "dfp_cia_aberta_BPP_con"      # Balanço Patrimonial Passivo Consolidado
    }
    
    def process_year(self, ano: str) -> Dict[str, bool]:
        """
        Processa todos os CSVs de um ano específico
        
        Args:
            ano: Ano a processar (ex: "2023")
            
        Returns:
            Dicionário com status de processamento de cada CSV
        """
        results = {}
        
        try:
            logger.info(f"🔄 Iniciando processamento Silver para ano {ano}")
            
            # 1. Buscar ZIP da camada Bronze
            zip_path = self._find_latest_zip(ano)
            if not zip_path:
                logger.error(f"❌ ZIP não encontrado para ano {ano}")
                return results
            
            # 2. Baixar ZIP do MinIO
            zip_bytes = self._download_from_minio(zip_path)
            
            # 3. Extrair e processar cada CSV
            with zipfile.ZipFile(BytesIO(zip_bytes)) as zf:
                for csv_key, csv_prefix in self.CSV_TARGETS.items():
                    csv_filename = f"{csv_prefix}_{ano}.csv"
                    
                    if csv_filename in zf.namelist():
                        logger.info(f"📄 Processando {csv_filename}...")
                        success = self._process_csv(zf, csv_filename, csv_key, ano)
                        results[csv_key] = success
                    else:
                        logger.warning(f"⚠️  {csv_filename} não encontrado no ZIP")
                        results[csv_key] = False
            
            logger.info(f"✅ Processamento concluído para {ano}")
            return results
            
        except Exception as e:
            logger.error(f"❌ Erro ao processar ano {ano}: {e}")
            return results
    
    def _find_latest_zip(self, ano: str) -> Optional[str]:
        """Encontra o ZIP mais recente para um ano na camada Bronze"""
        prefix = f"gov_br_cvm/demonstracoes_financeiras_padronizadas/ano={ano}/"
        
        try:
            objects = self.minio_client.list_objects(
                self.minio_config.bucket_bronze,
                prefix=prefix,
                recursive=True
            )
            
            zip_files = [obj.object_name for obj in objects if obj.object_name.endswith('.zip')]
            
            if not zip_files:
                return None
            
            # Retornar o mais recente (último da lista ordenada)
            return sorted(zip_files)[-1]
            
        except Exception as e:
            logger.error(f"❌ Erro ao buscar ZIP: {e}")
            return None
    
    def _download_from_minio(self, object_path: str) -> bytes:
        """Baixa objeto do MinIO e retorna bytes"""
        response = self.minio_client.get_object(
            self.minio_config.bucket_bronze,
            object_path
        )
        data = response.read()
        response.close()
        response.release_conn()
        return data
    
    def _process_csv(self, zf: zipfile.ZipFile, csv_filename: str, 
                     csv_key: str, ano: str) -> bool:
        """
        Processa um CSV específico do ZIP e salva como Delta Lake
        
        Args:
            zf: ZipFile aberto
            csv_filename: Nome do CSV dentro do ZIP
            csv_key: Chave identificadora (BP_con, DRE_con, etc)
            ano: Ano de referência
            
        Returns:
            True se processamento foi bem-sucedido
        """
        try:
            # 1. Ler CSV com Pandas (mais rápido para CSVs da CVM)
            with zf.open(csv_filename) as csv_file:
                df_pandas = pd.read_csv(
                    csv_file,
                    sep=';',
                    encoding='latin-1',
                    dtype=str  # Ler tudo como string primeiro
                )
            
            logger.info(f"  📊 Linhas lidas: {len(df_pandas):,}")
            
            # 2. Converter para Spark DataFrame
            df_spark = self.spark.createDataFrame(df_pandas)
            
            # 3. Aplicar transformações de limpeza
            df_clean = self._clean_dataframe(df_spark, csv_key)
            
            # 4. Adicionar colunas de metadata
            df_final = df_clean \
                .withColumn("ano_referencia", lit(int(ano))) \
                .withColumn("data_processamento", current_timestamp()) \
                .withColumn("fonte", lit("CVM")) \
                .withColumn("tipo_demonstracao", lit(csv_key))
            
            # 5. Salvar como Delta Lake
            delta_path = f"s3a://{self.minio_config.bucket_silver}/cvm/{csv_key}/"
            
            df_final.write \
                .format("delta") \
                .mode("overwrite") \
                .option("overwriteSchema", "true") \
                .partitionBy("ano_referencia") \
                .save(delta_path)
            
            logger.info(f"  ✅ Salvo em Delta: {delta_path}")
            
            # 6. Mostrar estatísticas
            total_rows = df_final.count()
            logger.info(f"  📈 Total de registros: {total_rows:,}")
            
            return True
            
        except Exception as e:
            logger.error(f"  ❌ Erro ao processar {csv_filename}: {e}")
            return False
    
    def _clean_dataframe(self, df, csv_key: str):
        """
        Aplica transformações de limpeza e padronização no DataFrame
        Args:
            df: DataFrame original lido do CSV
            csv_key: Chave identificadora para aplicar filtros específicos
        Returns: DataFrame limpo e padronizado

        """
        df_clean = df

        # Filter por DS_CONTA — apenas para tipos relevantes
        ds_conta_filters = {
            "DRE_con": r"lucro por ação|dividendos",
            "BPA_con": r"ativo circulante",
            "BPP_con": r"passivo circulante",
        }

        if csv_key in ds_conta_filters and "DS_CONTA" in df_clean.columns:
            df_clean = df_clean.filter(
                lower(col("DS_CONTA")).rlike(ds_conta_filters[csv_key])
            )

        # Filter COLUNA_DF — apenas para DMPL
        if csv_key == "DFC_DMPL_con" and "COLUNA_DF" in df_clean.columns:
            df_clean = df_clean.filter(
                lower(col("COLUNA_DF")).rlike(r"patrimônio líquido")
            )

        # Trimmar strings
        string_cols = [field.name for field in df.schema.fields
                    if isinstance(field.dataType, StringType)]
        for col_name in string_cols:
            df_clean = df_clean.withColumn(col_name, trim(col(col_name)))

        # Converter VL_CONTA com escala
        df_clean = df_clean.withColumn(
            "VL_CONTA",
            when(col("ESCALA_MOEDA") == "MIL", col("VL_CONTA").cast(DoubleType()) * 1000)
            .otherwise(col("VL_CONTA").cast(DoubleType()))
        )

        # Converter datas
        date_cols = ['DT_REFER', 'DT_INI_EXERC', 'DT_FIM_EXERC']
        for col_name in date_cols:
            if col_name in df_clean.columns:
                df_clean = df_clean.withColumn(
                    col_name,
                    to_date(col(col_name), "yyyy-MM-dd")
                )

        # Remover linhas completamente nulas
        df_clean = df_clean.dropna(how='all')

        # ⚠️ Guardar se DataFrame está vazio antes de retornar
        count = df_clean.count()
        if count == 0:
            logger.warning(f"⚠️ DataFrame vazio após filtros para {csv_key}!")
        else:
            logger.info(f"  ✅ {count:,} linhas após limpeza")

        return df_clean
        
    def read_silver_table(self, table_name: str, ano: Optional[int] = None):
        """
        Lê uma tabela Delta da camada Silver
        
        Args:
            table_name: Nome da tabela (BP_con, DRE_con, etc)
            ano: Ano específico (opcional, None = todos os anos)
            
        Returns:
            Spark DataFrame
        """
        delta_path = f"s3a://{self.minio_config.bucket_silver}/cvm/{table_name}/"
        
        df = self.spark.read.format("delta").load(delta_path)
        
        if ano:
            df = df.filter(col("ano_referencia") == ano)
        
        return df


processor = CVMSilverProcessor()

# Processar múltiplos anos
#anos = ["2020", "2021", "2022", "2023"]
anos = ["2023"]  # Para teste rápido, processar apenas 2025
for ano in anos:
    results = processor.process_year(ano)
    print(f"\n📊 Resultados para {ano}:")
    for csv_key, success in results.items():
        status = "✅" if success else "❌"
        print(f"  {status} {csv_key}")

"""# Exemplo de leitura
print("\n📖 Lendo tabela BP_con de 2023:")
df = processor.read_silver_table("BP_con", ano=2023)
df.show(5)
print(f"Total de registros: {df.count():,}")"""

In [ ]:
df = processor.read_silver_table("BPA_con", ano=2023)
df.show(5)


In [ ]:
# silver_quality.py
"""
Validação de qualidade de dados + Métricas Prometheus
"""

from pyspark.sql.functions import col, count, when, isnan, isnull, sum as spark_sum
from prometheus_client import Gauge, Counter, Histogram, generate_latest
import logging
from typing import Dict, Any

logger = logging.getLogger(__name__)

# ============================================================================
# MÉTRICAS PROMETHEUS
# ============================================================================

# Gauges (valores instantâneos)
silver_rows_total = Gauge(
    'silver_rows_total', 
    'Total de linhas na camada Silver',
    ['csv_type', 'ano']
)

silver_companies_unique = Gauge(
    'silver_companies_unique',
    'Número de empresas únicas',
    ['csv_type', 'ano']
)

silver_null_percentage = Gauge(
    'silver_null_percentage',
    'Percentual de valores nulos por coluna',
    ['csv_type', 'ano', 'column']
)

# Counters (valores acumulativos)
silver_validation_errors = Counter(
    'silver_validation_errors_total',
    'Total de erros de validação',
    ['csv_type', 'ano', 'error_type']
)

# Histograms (distribuições)
silver_processing_time = Histogram(
    'silver_processing_seconds',
    'Tempo de processamento em segundos',
    ['csv_type', 'ano']
)


class SilverQualityValidator:
    """Validador de qualidade de dados para camada Silver"""
    
    def __init__(self, spark):
        self.spark = spark
    
    def validate_and_report_metrics(self, df, csv_key: str, ano: str) -> Dict[str, Any]:
        """
        Executa validações E atualiza métricas Prometheus
        
        Returns:
            Dict com métricas de qualidade
        """
        
        logger.info(f"\n{'='*60}")
        logger.info(f"🔍 Validação de Qualidade: {csv_key} - {ano}")
        logger.info(f"{'='*60}")
        
        metrics = {
            "csv_key": csv_key,
            "ano": ano,
            "total_rows": df.count(),
            "total_columns": len(df.columns),
            "validations": {}
        }
        
        # ====================================================================
        # 1. COLUNAS OBRIGATÓRIAS
        # ====================================================================
        required_cols = {
            "DRE_con": ["CNPJ_CIA", "DENOM_CIA", "VL_CONTA", "DT_REFER", "DS_CONTA"],
            "BPA_con": ["CNPJ_CIA", "DENOM_CIA", "VL_CONTA", "CD_CONTA", "DS_CONTA"],
            "BPP_con": ["CNPJ_CIA", "DENOM_CIA", "VL_CONTA", "CD_CONTA", "DS_CONTA"],
            "DFC_DMPL_con": ["CNPJ_CIA", "DENOM_CIA", "VL_CONTA", "COLUNA_DF"]
        }
        
        if csv_key in required_cols:
            missing = [col for col in required_cols[csv_key] if col not in df.columns]
            metrics["validations"]["missing_columns"] = missing
            
            if missing:
                logger.error(f"❌ Colunas obrigatórias faltando: {missing}")
                silver_validation_errors.labels(
                    csv_type=csv_key, 
                    ano=ano, 
                    error_type='missing_columns'
                ).inc(len(missing))
                return metrics
            else:
                logger.info("✅ Todas as colunas obrigatórias presentes")
        
        # ====================================================================
        # 2. TOTAL DE REGISTROS (atualizar Prometheus)
        # ====================================================================
        silver_rows_total.labels(csv_type=csv_key, ano=ano).set(metrics["total_rows"])
        logger.info(f"📊 Total de registros: {metrics['total_rows']:,}")
        
        # ====================================================================
        # 3. EMPRESAS ÚNICAS
        # ====================================================================
        if "CNPJ_CIA" in df.columns:
            unique_companies = df.select("CNPJ_CIA").distinct().count()
            metrics["validations"]["unique_companies"] = unique_companies
            
            # Atualizar Prometheus
            silver_companies_unique.labels(csv_type=csv_key, ano=ano).set(unique_companies)
            
            logger.info(f"🏢 Empresas únicas: {unique_companies:,}")
        
        # ====================================================================
        # 4. PERCENTUAL DE NULOS POR COLUNA
        # ====================================================================
        null_percentages = {}
        high_null_columns = []
        
        for col_name in df.columns:
            null_count = df.filter(col(col_name).isNull()).count()
            null_pct = (null_count / metrics["total_rows"]) * 100
            null_percentages[col_name] = round(null_pct, 2)
            
            # Atualizar Prometheus
            silver_null_percentage.labels(
                csv_type=csv_key, 
                ano=ano, 
                column=col_name
            ).set(null_pct)
            
            # Alertar se > 50% nulos
            if null_pct > 50:
                logger.warning(f"⚠️  {col_name}: {null_pct:.1f}% nulos")
                high_null_columns.append(col_name)
                silver_validation_errors.labels(
                    csv_type=csv_key,
                    ano=ano,
                    error_type='high_null_percentage'
                ).inc()
        
        metrics["validations"]["null_percentages"] = null_percentages
        metrics["validations"]["high_null_columns"] = high_null_columns
        
        # ====================================================================
        # 5. VALORES MONETÁRIOS INVÁLIDOS
        # ====================================================================
        if "VL_CONTA" in df.columns:
            invalid_values = df.filter(
                col("VL_CONTA").isNull() | isnan(col("VL_CONTA"))
            ).count()
            
            invalid_pct = (invalid_values / metrics["total_rows"]) * 100
            metrics["validations"]["invalid_monetary_values"] = invalid_values
            metrics["validations"]["invalid_monetary_pct"] = round(invalid_pct, 2)
            
            if invalid_values > 0:
                logger.warning(f"⚠️  Valores monetários inválidos: {invalid_values:,} ({invalid_pct:.1f}%)")
                silver_validation_errors.labels(
                    csv_type=csv_key,
                    ano=ano,
                    error_type='invalid_monetary_values'
                ).inc(invalid_values)
            else:
                logger.info("✅ Todos os valores monetários são válidos")
        
        # ====================================================================
        # 6. DATAS NO FUTURO
        # ====================================================================
        if "DT_REFER" in df.columns:
            from datetime import date
            from pyspark.sql.functions import lit
            
            future_dates = df.filter(col("DT_REFER") > lit(date.today())).count()
            metrics["validations"]["future_dates"] = future_dates
            
            if future_dates > 0:
                logger.warning(f"⚠️  Datas no futuro: {future_dates:,}")
                silver_validation_errors.labels(
                    csv_type=csv_key,
                    ano=ano,
                    error_type='future_dates'
                ).inc(future_dates)
            else:
                logger.info("✅ Todas as datas são válidas")
        
        # ====================================================================
        # 7. DUPLICATAS
        # ====================================================================
        if csv_key in required_cols:
            # Definir chave única baseada nas colunas obrigatórias
            key_cols = ["CNPJ_CIA", "CD_CONTA", "DT_REFER"] if "CD_CONTA" in df.columns else ["CNPJ_CIA", "DT_REFER"]
            key_cols = [c for c in key_cols if c in df.columns]
            
            if key_cols:
                duplicates = df.groupBy(key_cols).count().filter(col("count") > 1).count()
                metrics["validations"]["duplicates"] = duplicates
                
                if duplicates > 0:
                    logger.warning(f"⚠️  Registros duplicados: {duplicates:,}")
                    silver_validation_errors.labels(
                        csv_type=csv_key,
                        ano=ano,
                        error_type='duplicates'
                    ).inc(duplicates)
                else:
                    logger.info("✅ Nenhuma duplicata encontrada")
        
        # ====================================================================
        # 8. DISTRIBUIÇÃO DE VALORES
        # ====================================================================
        if "VL_CONTA" in df.columns:
            stats = df.select(
                spark_sum("VL_CONTA").alias("total"),
                count("VL_CONTA").alias("count"),
                avg("VL_CONTA").alias("avg"),
                min("VL_CONTA").alias("min"),
                max("VL_CONTA").alias("max")
            ).collect()[0]
            
            metrics["validations"]["value_distribution"] = {
                "total": float(stats["total"]) if stats["total"] else 0,
                "count": stats["count"],
                "avg": float(stats["avg"]) if stats["avg"] else 0,
                "min": float(stats["min"]) if stats["min"] else 0,
                "max": float(stats["max"]) if stats["max"] else 0
            }
            
            logger.info(f"💰 Distribuição de valores:")
            logger.info(f"   Total: R$ {stats['total']:,.2f}" if stats["total"] else "   Total: N/A")
            logger.info(f"   Média: R$ {stats['avg']:,.2f}" if stats["avg"] else "   Média: N/A")
            logger.info(f"   Min: R$ {stats['min']:,.2f}" if stats["min"] else "   Min: N/A")
            logger.info(f"   Max: R$ {stats['max']:,.2f}" if stats["max"] else "   Max: N/A")
        
        # ====================================================================
        # 9. RESUMO FINAL
        # ====================================================================
        self._print_summary(metrics)
        
        return metrics
    
    def _print_summary(self, metrics: Dict[str, Any]):
        """Imprime resumo da validação"""
        
        logger.info(f"\n{'='*60}")
        logger.info(f"📋 RESUMO DA VALIDAÇÃO")
        logger.info(f"{'='*60}")
        logger.info(f"CSV: {metrics['csv_key']}")
        logger.info(f"Ano: {metrics['ano']}")
        logger.info(f"Total de registros: {metrics['total_rows']:,}")
        logger.info(f"Total de colunas: {metrics['total_columns']}")
        
        validations = metrics["validations"]
        
        if "unique_companies" in validations:
            logger.info(f"Empresas únicas: {validations['unique_companies']:,}")
        
        if "invalid_monetary_values" in validations:
            logger.info(f"Valores inválidos: {validations['invalid_monetary_values']:,}")
        
        if "duplicates" in validations:
            logger.info(f"Duplicatas: {validations['duplicates']:,}")
        
        if "future_dates" in validations:
            logger.info(f"Datas futuras: {validations['future_dates']:,}")
        
        if "high_null_columns" in validations and validations["high_null_columns"]:
            logger.info(f"Colunas com >50% nulos: {len(validations['high_null_columns'])}")
        
        logger.info(f"{'='*60}\n")


# ============================================================================
# FUNÇÃO PARA EXPOR MÉTRICAS PROMETHEUS
# ============================================================================

def export_prometheus_metrics(port: int = 8000):
    """
    Inicia servidor HTTP para expor métricas Prometheus
    
    Args:
        port: Porta para expor métricas (padrão: 8000)
    """
    from prometheus_client import start_http_server
    import time
    
    logger.info(f"📊 Iniciando servidor de métricas Prometheus na porta {port}")
    start_http_server(port)
    
    logger.info(f"✅ Métricas disponíveis em: http://localhost:{port}/metrics")
    
    # Manter servidor rodando
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        logger.info("🛑 Servidor de métricas encerrado")


if __name__ == "__main__":
    # Exemplo de uso
    from silver import CVMSilverProcessor
    
    processor = CVMSilverProcessor()
    validator = SilverQualityValidator(processor.spark)
    
    # Ler dados
    df = processor.read_silver_table("DRE_con", ano=2023)
    
    # Validar e gerar métricas
    metrics = validator.validate_and_report_metrics(df, "DRE_con", "2023")
    
    # Exportar métricas Prometheus (em background)
    import threading
    metrics_thread = threading.Thread(target=export_prometheus_metrics, daemon=True)
    metrics_thread.start()
    
    print("\n✅ Métricas disponíveis em http://localhost:8000/metrics")
    print("Pressione Ctrl+C para encerrar...")
    
    import time
    try:
        while True:
            time.sleep(1)
    except KeyboardInterrupt:
        pass